In [1]:
import sys
sys.path.append("../")

from qmpsqsc.models.mpsqsc.mpstate import MPState
from qmpsqsc.models.mpsqsc.tenpy import to_tenpy_mps, tenpy_to_mpstate
from tenpy.algorithms.mps_common import VariationalCompression
import numpy as np
import torch

In [2]:
d = 2
L = 10
chi = 10
mpstate = MPState(L=L, d=d, chi=chi, dtype=torch.float64)
mpstate.normalize(inplace=True)

mpstate2 = MPState(L=L, d=d, chi=chi, dtype=torch.float64)
mpstate2.normalize(inplace=True)

psi = to_tenpy_mps(mpstate)
psi2 = to_tenpy_mps(mpstate2)

mpstate2.overlap(mpstate) - psi2.overlap(psi)

/Users/keisuke/miniconda3/envs/mpsqsc/lib/python3.11/site-packages/tenpy/networks/mps.py:1629: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width)


tensor(-4.1633e-17, dtype=torch.float64)

In [3]:
target_chi = 5
max_sweeps = 100
tol_theta_diff = 1e-8
options = {
        "trunc_params": {
            "chi_max": int(target_chi),
        },
        "max_trunc_err" : 1.0,
        "max_sweeps": int(max_sweeps),
        "min_sweeps": 1,
        "tol_theta_diff": float(tol_theta_diff),
    }
psi_t = psi.copy()
engine = VariationalCompression(psi_t, options)
res = engine.run()  # modifies `psi` in place to its best χ≤target_chi approximation

psi_t.norm = 1

In [4]:
psi_t.overlap(psi)

np.float64(0.9499811407122245)

In [5]:
mpstate3 = tenpy_to_mpstate(psi_t)
mpstate3.normalize(inplace=True)

In [7]:
mpstate3.overlap(mpstate).item() 

0.9499811407122241